# 면접 코칭 피드백 생성기 (이형 스타일)

**사용 방법**
1. 셀을 순서대로 실행 (Shift+Enter)
2. STEP 4 셀에서 면접 질문과 답변을 입력
3. 피드백 생성 실행 → `feedback/` 폴더에 자동 저장

**전제 조건:** `01_rag_build.ipynb` 먼저 실행 → `chroma_db/` 폴더 생성 필요  
**API 키:** 프로젝트 폴더의 `.env` 파일에 `GOOGLE_API_KEY` 입력

In [ ]:
%pip install google-genai chromadb==0.6.3 sentence-transformers python-dotenv --quiet

In [ ]:
import os, json, time
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# ─── 경로 설정 ────────────────────────────────────────────────
BASE         = Path(r'C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)')
CHROMA_DIR   = BASE / 'chroma_db'
FEEDBACK_DIR = BASE / 'feedback'
FEEDBACK_DIR.mkdir(exist_ok=True)

# ─── .env에서 API 키 로드 ─────────────────────────────────────
load_dotenv(BASE / '.env')
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')

if not GOOGLE_API_KEY:
    raise ValueError(
        f'.env 파일에 GOOGLE_API_KEY가 없습니다.\n파일 위치: {BASE / ".env"}'
    )

MODEL = 'gemini-flash-lite-latest'   # 무료 1500회/일
print(f'설정 완료  |  모델: {MODEL}')

In [ ]:
# ─── Gemini 클라이언트 + 재시도 래퍼 ─────────────────────────
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

def gemini_call(contents, temperature=0.4, max_tokens=800, max_retries=5):
    """503/429 오류 시 자동 재시도"""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    temperature=temperature,
                    max_output_tokens=max_tokens
                )
            )
        except Exception as e:
            code = getattr(e, 'status_code', 0)
            if code in (503, 429) and attempt < max_retries - 1:
                wait = 30 if code == 429 else 2 ** attempt
                print(f'  [{code}] {wait}초 후 재시도... ({attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise

test = gemini_call('안녕. 한 문장으로만 답해.')
print('Gemini 연결 성공:', test.text[:50])

---
## STEP 1 — RAG 로드 (ChromaDB)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
collection = chroma_client.get_collection('interview_rag', embedding_function=ef)
print(f'RAG 로드 완료: {collection.count()}개 청크')

def retrieve(question, answer, n=4):
    """질문+답변으로 관련 청크 검색"""
    query = f"{question} {answer[:150]}"
    results = collection.query(query_texts=[query], n_results=n)
    docs = results['documents'][0]
    sources = [m.get('source', '') for m in results['metadatas'][0]]
    return docs, sources

---
## STEP 2 — 이형 스타일 프롬프트 설정

In [ ]:
SYSTEM_PROMPT = """당신은 대기업 인사담당자 출신의 직설적인 면접 코치 '이형'이다.
10년간 수천 명을 면접했고, 지원자 답변을 들으면 합격/불합격이 바로 보인다.
친근한 말투를 쓰되 평가는 냉정하게 한다.
제공된 참고 자료를 반드시 근거로 활용해 구체적인 피드백을 준다.
반드시 아래 형식으로만 답한다:

[이형의 팩폭 한줄평]
한 문장으로 이 답변의 핵심 문제 또는 강점을 직격한다.

[이형의 시선]
면접관 관점에서 이 답변이 어떻게 들리는지, 왜 좋은지/나쁜지 구체적으로 분석한다. (3~5문장)

[이형의 합격 처방전]
1. 즉시 실천 가능한 구체적 개선 방법
2. 답변 구조/내용 개선 방법
3. 면접관에게 어필할 포인트"""


def build_prompt(question, answer, contexts):
    context_text = '\n\n'.join(f'[참고{i+1}] {c}' for i, c in enumerate(contexts))
    return f"""{SYSTEM_PROMPT}

면접 질문: {question}

지원자 답변: {answer}

--- 참고 자료 (면접왕 이형 채널) ---
{context_text}

위 답변에 대해 이형 스타일로 피드백하라."""


print('프롬프트 설정 완료')

---
## STEP 3 — 피드백 함수

In [ ]:
def get_feedback(question: str, answer: str, verbose: bool = True) -> dict:
    """
    질문 + 지원자 답변을 받아 이형 스타일 피드백 반환
    반환: {'question', 'answer', 'feedback', 'sources', 'timestamp'}
    """
    if verbose:
        print('RAG 검색 중...')
    contexts, sources = retrieve(question, answer)

    if verbose:
        print(f'검색 완료: {len(contexts)}개 청크 | 출처: {sources}')
        print('피드백 생성 중...\n')

    prompt = build_prompt(question, answer, contexts)
    resp = gemini_call(prompt, temperature=0.4, max_tokens=800)
    feedback = resp.text

    if verbose:
        print('━' * 60)
        print(feedback)
        print('━' * 60)

    return {
        'question':  question,
        'answer':    answer,
        'feedback':  feedback,
        'sources':   sources,
        'timestamp': datetime.now().isoformat()
    }

print('피드백 함수 준비 완료')

---
## STEP 4 — 피드백 받기

> **여기서 질문과 답변을 수정하고 셀을 실행하세요.**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ▼ 여기를 수정하세요
QUESTION = """1분 자기소개를 해보세요."""

ANSWER = """안녕하세요. 저는 성실하고 책임감 있는 사람입니다.
학교에서 열심히 공부했고, 팀 프로젝트도 많이 해봤습니다.
이 회사에 오고 싶어서 지원했습니다. 잘 부탁드립니다."""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

result = get_feedback(QUESTION, ANSWER)

---
## STEP 5 — 결과 저장

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = FEEDBACK_DIR / f'feedback_{ts}.json'

with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f'저장 완료: {save_path}')

---
## STEP 6 — 여러 질문 연속 처리 (선택)

여러 Q&A를 한 번에 돌리고 싶을 때 사용.

In [ ]:
# 여러 Q&A 목록 — 필요에 따라 추가/수정
qa_list = [
    {
        'question': '본인의 장점과 단점을 말해보세요.',
        'answer': '장점은 성실함이고 단점은 완벽주의입니다. 너무 꼼꼼하게 하다보면 시간이 걸리기도 합니다.'
    },
    {
        'question': '왜 이 회사에 지원했나요?',
        'answer': '연봉이 좋고 대기업이라 안정적일 것 같아서 지원했습니다. 복지도 좋다고 들었습니다.'
    },
]

batch_results = []
for i, qa in enumerate(qa_list):
    print(f'\n[{i+1}/{len(qa_list)}] 처리 중: {qa["question"][:30]}...')
    r = get_feedback(qa['question'], qa['answer'], verbose=False)
    batch_results.append(r)
    print(r['feedback'][:200], '...')
    if i < len(qa_list) - 1:
        time.sleep(5)   # 15 RPM 한도 대비

# 배치 저장
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
batch_path = FEEDBACK_DIR / f'batch_{ts}.json'
with open(batch_path, 'w', encoding='utf-8') as f:
    json.dump(batch_results, f, ensure_ascii=False, indent=2)

print(f'\n배치 저장 완료: {batch_path}  ({len(batch_results)}건)')